In [1]:
import sys, os
sys.path.insert(0, 'src')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
from data_loader import load_raw_data, get_dataset_metadata
from preprocessing import preprocess_pipeline
raw_df = load_raw_data()
metadata = get_dataset_metadata(raw_df)
print('Raw Dataset Metadata:', metadata)
clean_df, sensor_report = preprocess_pipeline(raw_df)
print(f'Preprocessed Dataset: {len(clean_df):,} rows x {clean_df.shape[1]} columns')
print('Sensor Anomaly Treatment Report:', sensor_report)


[DataLoader] Loading dataset from: C:\Users\umars\OneDrive\Desktop\PMAccelerator\data\GlobalWeatherRepository.csv
[DataLoader] Successfully loaded 98,604 records across 41 columns.
Raw Dataset Metadata: {'total_rows': 98604, 'total_columns': 41, 'memory_usage_mb': np.float64(40.74), 'unique_countries': 211, 'unique_locations': 254, 'date_min': '2024-05-16 01:45', 'date_max': '2025-10-05 19:30', 'null_count': 0}
[Preprocessing] Starting data cleaning and feature engineering pipeline...
[Preprocessing] Completed pipeline. Total columns now: 75
Preprocessed Dataset: 98,604 rows x 75 columns
Sensor Anomaly Treatment Report: {'air_quality_Carbon_Monoxide_negative_cleaned': 1, 'air_quality_Ozone_negative_cleaned': 0, 'air_quality_Nitrogen_dioxide_negative_cleaned': 0, 'air_quality_Sulphur_dioxide_negative_cleaned': 1, 'air_quality_PM2.5_negative_cleaned': 0, 'air_quality_PM10_negative_cleaned': 1}


In [3]:
from eda import run_full_eda
eda_results = run_full_eda(clean_df, output_dir='outputs/figures')
print('Statistical Profiling Summary:')
print(eda_results['summary_table'].head(10))


[EDA] Executing Exploratory Data Analysis suite...
[EDA] Saved: outputs/figures\eda_temp_distribution.png
[EDA] Saved: outputs/figures\eda_precip_distribution.png
[EDA] Saved: outputs/figures\eda_weather_correlation_matrix.png
[EDA] Saved: outputs/figures\eda_temporal_temperature_trend.png
Statistical Profiling Summary:
                       count         mean  ...      kurtosis     iqr
temperature_celsius  98604.0    22.763610  ...      0.891672  10.200
feels_like_celsius   98604.0    23.809104  ...      0.826327  12.700
precip_mm            98604.0     0.141398  ...    662.421410   0.030
humidity             98604.0    64.260385  ...     -0.650518  37.000
wind_kph             98604.0    13.248223  ...  30146.572703  11.900
pressure_mb          98604.0  1013.984240  ...  19190.908770   8.000
cloud                98604.0    39.361000  ...     -1.337728  75.000
visibility_km        98604.0     9.638864  ...     15.759093   0.000
uv_index             98604.0     3.864794  ...     -0.450

In [4]:
from anomaly_detection import run_full_anomaly_detection
anom_df, anom_results = run_full_anomaly_detection(clean_df, fig_dir='outputs/figures', metric_dir='outputs/metrics')
print('Detected Weather Anomalies Summary:')
for metric_name, val in anom_results['metrics'].items():
    print(f'  - {metric_name}: {val}')


[Anomaly] Initiating Multi-Method Anomaly Detection Pipeline...
[Anomaly] Saved: outputs/figures\anomaly_isolation_forest_scatter.png
[Anomaly] Saved: outputs/figures\anomaly_timeline.png
[Anomaly] Saved: outputs/figures\anomaly_top_locations.png
[Anomaly] Exported top anomalies table to: outputs/metrics\detected_anomalies_top100.csv
Detected Weather Anomalies Summary:
  - total_records: 98604
  - iso_forest_anomalies: 1973
  - iso_forest_anomaly_pct: 2.0
  - extreme_heat_days: 1281
  - extreme_cold_days: 175
  - extreme_precip_days: 181
  - extreme_wind_days: 1006
  - hazardous_air_days: 1439
  - any_extreme_event_days: 3947


In [5]:
from models import run_full_forecasting_pipeline
from evaluation import plot_model_comparison, plot_residuals, plot_actual_vs_predicted
metrics_table, predictions, model_artifacts = run_full_forecasting_pipeline(clean_df)
plot_model_comparison(metrics_table, output_dir='outputs/figures')
y_test = model_artifacts['y_test']
res_dict = {name: (y_test, preds) for name, preds in predictions.items() if 'Baseline' not in name}
plot_residuals(res_dict, output_dir='outputs/figures')
best_model = metrics_table.iloc[0]['model_name']
plot_actual_vs_predicted(y_test, predictions[best_model], model_name=best_model, output_dir='outputs/figures')
print('Model Evaluation Benchmark:')
print(metrics_table.to_string(index=False))


[Models] Initiating Multi-Model Forecasting Pipeline...
[Models] Chronological Split Date: 2025-06-26
[Models] Train size: 78,721 records (2024-05-16 to 2025-06-25)
[Models] Test size:  19,883 records (2025-06-26 to 2025-10-05)
[Models] Features used (23): ['temp_lag_1', 'temp_lag_2', 'temp_lag_3', 'temp_lag_7', 'temp_roll_mean_3', 'temp_roll_mean_7', 'temp_roll_std_7', 'temp_roll_min_7', 'temp_roll_max_7', 'temp_diff_1', 'humidity_lag_1', 'pressure_lag_1', 'wind_lag_1', 'precip_lag_1', 'latitude', 'longitude', 'sin_hour', 'cos_hour', 'sin_month', 'cos_month', 'sin_dayofyear', 'cos_dayofyear', 'is_weekend']
  -> Evaluating Persistence Baseline...
  -> Training Ridge Regression...
  -> Training Random Forest Regressor...
  -> Training Gradient Boosting Regressor...
[Models] Using Histogram Gradient Boosting Regressor...
  -> Training Deep Learning (PyTorch MLP Forecaster)...
  -> Building Meta-Ensemble Forecaster...
[Models] Meta-Ensemble Optimal Weights: {'Ridge Regression': 0.0564, 'R

C:\Users\umars\OneDrive\Desktop\PMAccelerator\src\evaluation.py:66: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
C:\Users\umars\OneDrive\Desktop\PMAccelerator\src\evaluation.py:66: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
C:\Users\umars\OneDrive\Desktop\PMAccelerator\src\evaluation.py:66: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")


In [6]:
from advanced_analyses import run_all_advanced_analyses
test_df = model_artifacts['test_df']
feat_cols = model_artifacts['feature_cols']
rf_model = model_artifacts['models']['RandomForest']
X_test = test_df[feat_cols].values
adv_results = run_all_advanced_analyses(
    clean_df, rf_model, feat_cols, X_test, y_test,
    fig_dir='outputs/figures', metric_dir='outputs/metrics'
)
print('All 5 Advanced Unique Analyses completed successfully!')


[Advanced Analyses] 1/5 Running Climate Analysis...
  -> Saved: outputs/figures\climate_zone_patterns.png
[Advanced Analyses] 2/5 Running Environmental Impact Analysis...
  -> Saved: outputs/figures\environmental_air_quality_correlations.png
[Advanced Analyses] 3/5 Running Feature Importance Analysis...
  -> Saved: outputs/figures\feature_importance_comparison.png
[Advanced Analyses] 4/5 Running Spatial Analysis...
  -> Saved: outputs/figures\spatial_latitude_gradient.png
[Advanced Analyses] 5/5 Running Geographical Patterns Analysis...
  -> Saved: outputs/figures\geographical_continental_patterns.png
All 5 Advanced Unique Analyses completed successfully!


C:\Users\umars\OneDrive\Desktop\PMAccelerator\src\advanced_analyses.py:297: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator. Otherwise, ticks may be mislabeled.
  axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=35, ha="right")
